# GNN Feature Extraction — BRACS
**AFFC-Net Base Paper Implementation**

Builds superpixel graphs, trains 3-layer GAT, extracts `[N, 196, 1024]` node features.

**Dataset:** BRACS (Breast Carcinoma Subtyping) — 7 classes across train / val / test splits.  
Classes: `0_N, 1_PB, 2_UDH, 3_FEA, 4_ADH, 5_DCIS, 6_IC`

**Method:** Identical to LC25000 GNN pipeline. Only the dataset class differs — BRACS uses
explicit `train/`, `val/`, `test/` sub-folders instead of a flat class-folder layout.

**Corrections (inherited from LC25000 notebook):**
- `.ipynb_checkpoints` and hidden dirs filtered out
- SLIC runs on CIE-Lab color space (paper §4.1.2)
- Node features: 5-dim `(x/224, y/224, R/224, G/224, B/224)` per paper Eq.6
- `num_classes=7` enforced; labels saved as clean `[0..6]`
- No augmentation at extraction time (deterministic, matches CNN notebook)

In [19]:
import os, glob, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, TopKPooling, global_mean_pool
from skimage.segmentation import slic
from skimage.color import rgb2lab
from skimage import graph as sk_graph

# ── GPU ────────────────────────────────────────────────────────────────────────
os.environ["CUDA_VISIBLE_DEVICES"] = "3"   # ← change to your GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))

Image.MAX_IMAGE_PIXELS = None


Device: cuda
GPU   : NVIDIA H200


## Superpixel graph construction
**Paper §4.1.2:** SLIC in CIE-Lab space, 300 segments. Node features = `(x/224, y/224, R/224, G/224, B/224)` (Eq.6). Edges from adjacency matrix Eq.5.

In [20]:
# ── Paper Eq.6: node feature construction ─────────────────────────────────────
def generate_superpixels(image_tensor):
    """
    image_tensor: [3, H, W] float32 in [0,1] (after ToTensor, no Normalize)
    Returns: segments array [H, W] with integer labels starting at 0
    Paper §4.1.2: SLIC in CIE-Lab color space, n_segments=300
    """
    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()  # [H, W, 3]
    img_lab = rgb2lab(img_np)                              # CIE-Lab conversion (paper)
    segments = slic(
        img_lab,
        n_segments=300,
        compactness=10,     # paper default
        sigma=1,
        start_label=0,
        channel_axis=-1
    )
    return segments


def create_node_features(image_tensor, segments):
    """
    Paper Eq.6: x_i = softmax([a_i/224, b_i/224, R_i/224, G_i/224, B_i/224])
    where a_i, b_i = centroid coordinates; R, G, B = mean color of segment.
    Returns: np.array [num_segments, 5]
    """
    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()  # [H, W, 3], float in [0,1]
    H, W   = img_np.shape[:2]
    nodes  = []
    for seg_id in sorted(np.unique(segments)):
        mask   = (segments == seg_id)
        coords = np.argwhere(mask)         # [[row, col], ...]
        # centroid: row=y, col=x  →  normalize by 224 (paper uses 224)
        y_c = coords[:, 0].mean() / 224.0
        x_c = coords[:, 1].mean() / 224.0
        color = img_np[mask].mean(axis=0)  # mean RGB in [0,1]
        # paper normalizes color by 224 (from Eq.6 denominator)
        feat = np.array([
            x_c,
            y_c,
            color[0] / 224.0 * 255.0,     # R/224 (img already in [0,1], ×255/224)
            color[1] / 224.0 * 255.0,
            color[2] / 224.0 * 255.0,
        ], dtype=np.float32)
        nodes.append(feat)
    nodes = np.array(nodes, dtype=np.float32)
    # Apply softmax per node (paper Eq.6)
    exp   = np.exp(nodes - nodes.max(axis=1, keepdims=True))
    nodes = exp / exp.sum(axis=1, keepdims=True)
    return nodes


def create_edges(segments):
    """
    Paper Eq.5: A_ij=1 if superpixels i and j are adjacent.
    Uses RAG (region adjacency graph) on the segment map.
    """
    unique = sorted(np.unique(segments))
    # Remap to 0-indexed with no gaps
    remap = np.zeros(segments.max() + 1, dtype=np.int32)
    for new, old in enumerate(unique):
        remap[old] = new
    seg_r = remap[segments]

    # Dummy RGB image for rag_mean_color (only topology matters here)
    dummy = np.zeros((*seg_r.shape, 3), dtype=np.float32)
    rag   = sk_graph.rag_mean_color(dummy, seg_r)

    edges = []
    for u, v in rag.edges:
        if u < len(unique) and v < len(unique):
            edges += [[u, v], [v, u]]
    if not edges:
        return torch.zeros((2, 0), dtype=torch.long)
    return torch.tensor(edges, dtype=torch.long).t().contiguous()


def image_to_graph(image_tensor, label):
    """Full pipeline: image → PyG Data object"""
    segments   = generate_superpixels(image_tensor)
    node_feats = create_node_features(image_tensor, segments)
    edge_index = create_edges(segments)
    return Data(
        x          = torch.tensor(node_feats, dtype=torch.float),
        edge_index = edge_index,
        y          = torch.tensor([label], dtype=torch.long)
    )

print("Graph construction functions defined.")
print("Node feature dim: 5  (x/224, y/224, R/224, G/224, B/224 — paper Eq.6)")


Graph construction functions defined.
Node feature dim: 5  (x/224, y/224, R/224, G/224, B/224 — paper Eq.6)


## Dataset — BRACS
BRACS has explicit `train/`, `val/`, `test/` sub-folders each containing 7 class sub-directories.
All three splits are loaded and concatenated (train → val → test) for graph pre-computation
and GNN training, identical to the LC25000 approach.

The `HistologyDataset` class is the **only** change vs the LC25000 notebook.
It uses `SPLITS = ["train", "val", "test"]` to walk the split-aware folder structure.

In [21]:
DATA_DIR = "BRACS_Dataset/BRACS_RoI/latest_version"   # ← change if needed

class HistologyDataset(Dataset):
    SPLITS   = ["train", "val", "test"]
    IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".bmp")

    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.images   = []
        self.labels   = []

        # No augmentation — deterministic extraction to match CNN notebook
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),   # paper: direct 224×224
            transforms.ToTensor()            # no Normalize — raw RGB needed for node features
        ])

        # ── Discover class names from the train split (canonical ordering) ─────
        train_split_dir = os.path.join(root_dir, "train")
        class_names = sorted([
            d for d in os.listdir(train_split_dir)
            if os.path.isdir(os.path.join(train_split_dir, d))
            and not d.startswith('.')   # skip hidden dirs (.ipynb_checkpoints etc.)
        ])
        self.class_to_idx = {cls: i for i, cls in enumerate(class_names)}
        print(f"Classes found: {self.class_to_idx}")

        # ── Walk all splits in fixed order: train → val → test ─────────────────
        for split in self.SPLITS:
            sp = os.path.join(root_dir, split)
            if not os.path.isdir(sp):
                continue
            for cls in os.listdir(sp):
                if cls.startswith('.'):
                    continue                          # skip hidden dirs
                cls_path = os.path.join(sp, cls)
                if not os.path.isdir(cls_path):
                    continue
                for file in os.listdir(cls_path):
                    if file.lower().endswith(self.IMG_EXTS):
                        self.images.append(os.path.join(cls_path, file))
                        self.labels.append(self.class_to_idx[cls])

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        img = self.transform(img)
        return image_to_graph(img, self.labels[idx])


raw_dataset = HistologyDataset(DATA_DIR)
NUM_CLASSES = len(raw_dataset.class_to_idx)
print(f"\nTotal images : {len(raw_dataset)}")
print(f"Num classes  : {NUM_CLASSES}")
assert NUM_CLASSES == 7, f"Expected 7 classes, got {NUM_CLASSES}. Check DATA_DIR structure."


Classes found: {'0_N': 0, '1_PB': 1, '2_UDH': 2, '3_FEA': 3, '4_ADH': 4, '5_DCIS': 5, '6_IC': 6}

Total images : 4168
Num classes  : 7


## Pre-compute and cache graphs
Processing 3892 images with SLIC takes ~10–20 min. Cache to disk to avoid re-doing.

In [22]:
CACHE_DIR = "graph_cache_bracs_ver2"

def precompute_graphs(dataset, cache_dir):
    os.makedirs(cache_dir, exist_ok=True)
    total = len(dataset)
    t0    = time.time()
    skipped = 0
    for idx in range(total):
        cache_path = os.path.join(cache_dir, f"graph_{idx:06d}.pt")
        if os.path.exists(cache_path):
            skipped += 1
            continue
        try:
            data = dataset[idx]
            torch.save(data, cache_path)
        except Exception as e:
            print(f"  Warning: skipping idx {idx} — {e}")
        if idx % 200 == 0:
            elapsed = time.time() - t0
            eta     = (elapsed / max(idx - skipped + 1, 1)) * (total - idx)
            print(f"  [{idx:>5}/{total}]  elapsed {elapsed/60:.1f}min  ETA {eta/60:.1f}min")
    print(f"Done. Total cached: {len(os.listdir(cache_dir))} files")

precompute_graphs(raw_dataset, CACHE_DIR)


Done. Total cached: 4159 files


In [23]:
class CachedDataset(Dataset):
    def __init__(self, cache_dir):
        self.files = sorted(glob.glob(os.path.join(cache_dir, "graph_*.pt")))
        print(f"Loaded {len(self.files)} cached graphs from {cache_dir}")
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        return torch.load(self.files[idx], weights_only=False)

cached_dataset = CachedDataset(CACHE_DIR)

# ── Verify labels are clean 0–6 ───────────────────────────────────────────────
print("\nVerifying labels...")
sample_labels = []
for f in sorted(glob.glob(os.path.join(CACHE_DIR, "graph_*.pt")))[:500]:
    d = torch.load(f, weights_only=False)
    sample_labels.append(d.y.item())
print(f"  Unique labels (first 500): {sorted(set(sample_labels))}")
assert max(sample_labels) <= 6, "Labels out of range — re-run dataset creation"
assert min(sample_labels) >= 0, "Negative label found"
print("  Label check passed.")

loader = DataLoader(
    cached_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True
)
print(f"DataLoader ready: {len(loader)} batches")


Loaded 4159 cached graphs from graph_cache_bracs_ver2

Verifying labels...
  Unique labels (first 500): [0, 4]
  Label check passed.
DataLoader ready: 130 batches


## GAT model
Paper §4.1.3: 3 GAT layers with 1, 2, 4 attention heads. Multi-head K=2N per layer. BatchNorm after each layer. TopKPooling to 196 nodes. Projection to 1024-dim (match CNN).

`num_classes=7` for BRACS (vs 5 for LC25000). All architecture parameters are identical.

In [24]:
class GNNModel(nn.Module):
    """
    Paper §4.1.3 GNN feature extractor:
      - 3-layer GAT (heads: 1, 2, 4 — matching K=2N per layer)
      - BatchNorm after each GAT layer
      - TopKPooling: 300 → 196 nodes (ratio=196/300)
      - Linear projection: 256 → 1024 (match CNN feature dim)
    Input node feature dim: 5 (paper Eq.6)
    """
    def __init__(self, in_features=5, num_classes=7, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        # Layer 1: 1 head, concat=False → out: 32
        self.gat1 = GATConv(in_features, 32,  heads=1, concat=False)
        self.bn1  = nn.BatchNorm1d(32)

        # Layer 2: 2 heads, concat=True → out: 64×2=128
        self.gat2 = GATConv(32, 64, heads=2, concat=True)
        self.bn2  = nn.BatchNorm1d(128)

        # Layer 3: 4 heads, concat=False → out: 256
        self.gat3 = GATConv(128, 256, heads=4, concat=False)
        self.bn3  = nn.BatchNorm1d(256)

        # TopKPooling: 300 → 196 nodes (to match CNN spatial shape 14×14=196)
        self.topk_pool = TopKPooling(in_channels=256, ratio=196/300)

        # Project 256 → 1024 to match CNN feature dimension for AFF
        self.proj = nn.Linear(256, 1024)

        # Classification head (used during GNN pre-training only)
        self.fc = nn.Linear(1024, num_classes)

    def forward(self, x, edge_index, batch, return_node_features=False):
        # GAT layer 1
        x = F.relu(self.bn1(self.gat1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        # GAT layer 2
        x = F.relu(self.bn2(self.gat2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        # GAT layer 3
        x = F.relu(self.bn3(self.gat3(x, edge_index)))

        # TopKPooling: 300 → 196 nodes
        x, edge_index, _, batch, _, _ = self.topk_pool(x, edge_index, batch=batch)

        # Project to 1024
        x = self.proj(x)   # [total_kept_nodes, 1024]

        if return_node_features:
            return x, batch   # return per-node features for feature extraction

        # Classification head
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.fc(x)


model = GNNModel(in_features=5, num_classes=NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=10, min_lr=1e-6
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {total_params:,}")
print(f"Input node features: 5 (paper Eq.6)")


GNNModel(
  (gat1): GATConv(5, 32, heads=1)
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (gat2): GATConv(32, 64, heads=2)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (gat3): GATConv(128, 256, heads=4)
  (bn3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (topk_pool): TopKPooling(256, ratio=0.6533333333333333, multiplier=1.0)
  (proj): Linear(in_features=256, out_features=1024, bias=True)
  (fc): Linear(in_features=1024, out_features=7, bias=True)
)

Trainable parameters: 409,543
Input node features: 5 (paper Eq.6)


## Train GNN
GNN is pre-trained on the full dataset for feature extraction. Best model saved by train accuracy.

In [25]:
EPOCHS          = 100
PATIENCE        = 20
best_acc        = 0.0
patience_ctr    = 0
best_model_path = "gnn_model_bracs_best_ver2.pth"

print(f"Training GNN for up to {EPOCHS} epochs (patience={PATIENCE})...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = correct = total = 0
    t0 = time.time()

    for i, data in enumerate(loader):
        data = data.to(device)
        optimizer.zero_grad()
        out  = model(data.x, data.edge_index, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        pred        = out.argmax(dim=1)
        correct    += (pred == data.y).sum().item()
        total      += data.y.size(0)

        if i % 100 == 0:
            print(f"  Ep {epoch} | Batch {i:>4}/{len(loader)} | "
                  f"loss={loss.item():.4f} | acc={correct/max(total,1):.3f}")

    train_acc = correct / total
    avg_loss  = total_loss / len(loader)
    scheduler.step(train_acc)
    elapsed   = time.time() - t0
    print(f"Epoch {epoch:03d} | Loss={avg_loss:.4f} | Acc={train_acc:.4f} | {elapsed:.1f}s")

    if train_acc > best_acc:
        best_acc     = train_acc
        patience_ctr = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✓ Best acc={best_acc:.4f} — saved")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest training accuracy: {best_acc:.4f}")
torch.save(model.state_dict(), "gnn_model_bracs_final_ver2.pth")


Training GNN for up to 100 epochs (patience=20)...
  Ep 1 | Batch    0/130 | loss=1.9522 | acc=0.156
  Ep 1 | Batch  100/130 | loss=1.7904 | acc=0.251
Epoch 001 | Loss=1.8136 | Acc=0.2592 | 14.2s
  ✓ Best acc=0.2592 — saved
  Ep 2 | Batch    0/130 | loss=2.0280 | acc=0.125
  Ep 2 | Batch  100/130 | loss=1.6018 | acc=0.327
Epoch 002 | Loss=1.7001 | Acc=0.3323 | 10.7s
  ✓ Best acc=0.3323 — saved
  Ep 3 | Batch    0/130 | loss=1.6972 | acc=0.312
  Ep 3 | Batch  100/130 | loss=1.3273 | acc=0.365
Epoch 003 | Loss=1.6297 | Acc=0.3669 | 12.3s
  ✓ Best acc=0.3669 — saved
  Ep 4 | Batch    0/130 | loss=1.4659 | acc=0.406
  Ep 4 | Batch  100/130 | loss=1.5577 | acc=0.386
Epoch 004 | Loss=1.5902 | Acc=0.3789 | 13.7s
  ✓ Best acc=0.3789 — saved
  Ep 5 | Batch    0/130 | loss=1.7180 | acc=0.344
  Ep 5 | Batch  100/130 | loss=1.7621 | acc=0.383
Epoch 005 | Loss=1.5609 | Acc=0.3857 | 12.0s
  ✓ Best acc=0.3857 — saved
  Ep 6 | Batch    0/130 | loss=1.3778 | acc=0.531
  Ep 6 | Batch  100/130 | loss=1.2

## Extract and save node features `[N, 196, 1024]`
Load best model, run in `return_node_features=True` mode, pad/trim to exactly 196 nodes per image.

In [26]:
MAX_NODES = 196     # matches CNN spatial shape 14×14
FEAT_DIM  = 1024    # matches CNN channel dim

model.load_state_dict(torch.load(best_model_path, weights_only=True))
model.eval()

all_node_features = []
all_labels        = []

print("Extracting GNN node features...")
with torch.no_grad():
    for batch_idx, data in enumerate(loader):
        data       = data.to(device)
        node_feats, batch_vec = model(
            data.x, data.edge_index, data.batch,
            return_node_features=True
        )
        labels_b   = data.y
        batch_size = labels_b.size(0)

        for img_idx in range(batch_size):
            mask      = (batch_vec == img_idx)
            img_nodes = node_feats[mask]          # [n_kept, 1024]
            n         = img_nodes.size(0)

            if n < MAX_NODES:
                pad       = torch.zeros(MAX_NODES - n, FEAT_DIM, device=device)
                img_nodes = torch.cat([img_nodes, pad], dim=0)
            else:
                img_nodes = img_nodes[:MAX_NODES]  # [196, 1024]

            all_node_features.append(img_nodes.cpu())
        all_labels.extend(labels_b.cpu().tolist())

        if batch_idx % 50 == 0:
            print(f"  Batch {batch_idx}/{len(loader)}")

gnn_features  = torch.stack(all_node_features)   # [N, 196, 1024]
labels_tensor = torch.tensor(all_labels, dtype=torch.long)

print(f"\nGNN feature shape : {gnn_features.shape}")
print(f"Labels shape      : {labels_tensor.shape}")
print(f"Unique labels     : {torch.unique(labels_tensor).tolist()}")
assert gnn_features.shape[1:] == (196, 1024), "Shape mismatch!"
assert labels_tensor.min() >= 0 and labels_tensor.max() <= 6, "Label range error!"
print("All assertions passed.")


Extracting GNN node features...
  Batch 0/130
  Batch 50/130
  Batch 100/130

GNN feature shape : torch.Size([4159, 196, 1024])
Labels shape      : torch.Size([4159])
Unique labels     : [0, 1, 2, 3, 4, 5, 6]
All assertions passed.


In [27]:
torch.save(gnn_features,  "bracs_gnn_features_ver2.pt")
torch.save(labels_tensor, "bracs_labels_ver2.pt")   # ← authoritative label file
print(f"Saved → bracs_gnn_features_ver2.pt  {gnn_features.shape}")
print(f"Saved → bracs_labels_ver2.pt        {labels_tensor.shape}")
print(f"Class mapping: {raw_dataset.class_to_idx}")


Saved → bracs_gnn_features_ver2.pt  torch.Size([4159, 196, 1024])
Saved → bracs_labels_ver2.pt        torch.Size([4159])
Class mapping: {'0_N': 0, '1_PB': 1, '2_UDH': 2, '3_FEA': 3, '4_ADH': 4, '5_DCIS': 5, '6_IC': 6}
